# 🤖 UPPER CONFIDENCE BOUND (UCB)

> **🎯 AMAÇ:** Keşfetme-Sömürme (Exploration-Exploitation) dengesini kurarak en yüksek ödülü getiren seçeneği bulmak.  
> **📥 GİRDİ:** Ortam Verisi (Örn: Tıklama Durumları - 1 veya 0)  
> **📤 ÇIKTI:** Toplam Ödül, Seçilen İlanların Dağılımı  

### ⏱️ NE ZAMAN KULLANILIR?
- Dijital Reklam Optimizasyonu (Hangi reklama tıklanıyor?)
- A/B Testlerinde dinamik seçim
- Slot Makinesi (Multi-Armed Bandit) problemleri

### ⚠️ UYARI
Pekiştirmeli öğrenmede **X_train** (sabit veri) yerine bir **Environment (Ortam)** vardır.
Model bir eylem yapar (Ad 1'i göster), ortam bir cevap döner (Tıklandı: 1, Tıklanmadı: 0).

Bu şablonda **`df` değişkeni**, ortamın kendisini simüle eder (Ground Truth).
Yani model "3. ilanı seçtim" dediğinde, `df` tablosuna bakıp "O sırada 3. ilana tıklanmış mı?" cevabını alırız.

In [ ]:
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

print("=" * 40)
print("🤖 UCB ALGORİTMASI BAŞLATILIYOR")
print("=" * 40)

In [ ]:
# 1. DEĞİŞKENLERİN VE ORTAMIN HAZIRLANMASI
# ---------------------------------------------------
# Simülasyon verisi (Gerçek hayatta burası canlı veri akışı olur)
# Örnek: pd.read_csv('Ads_CTR_Optimisation.csv')
# Biz burada df değişkeninin dışarıdan geldiğini varsayıyoruz.

if 'df' not in locals():
    print("⚠️ UYARI: 'df' değişkeni bulunamadı! Lütfen veriyi yükleyin.")
    print("Örnek: df = pd.read_csv('veriler.csv')")
else:
    print(f"✅ Veri Hazır. Toplam Satır: {len(df)}")

# <--- LÜTFEN BURAYI DÜZENLEYİN
N = 1000 # Toplam işlem sayısı (Simülasyon döngüsü - Kaç kullanıcı gelecek?)
d = 10   # Toplam seçenek sayısı (Kaç farklı reklam var?)

# UCB için gerekli listeler
oduller = [0] * d     # Ni: Her bir ilanın kaç kez seçildiği
tiklamalar = [0] * d  # Ri: Her bir ilandan gelen toplam ödül
toplam = 0            # Toplam kazanılan ödül
secilenler = []       # Her adımda hangi ilanın seçildiği

In [ ]:
# 2. UCB ALGORİTMASI (DÖNGÜ)
# ---------------------------------------------------
print("\n🚀 Simülasyon Başlıyor... (Model öğreniyor)")

for n in range(0, N):
    ad = 0          # O turda seçilecek ilan indeksi
    max_ucb = 0     # O turdaki maksimum UCB değeri
    
    # Her bir ilanı gez ve UCB değerini hesapla
    for i in range(0, d):
        if tiklamalar[i] > 0:
            # 1. Aşama: O ana kadarki ortalama başarı (Exploitation)
            ortalama = oduller[i] / tiklamalar[i]
            
            # 2. Aşama: Belirsizlik / Keşfetme payı (Exploration)
            # math.log(n + 1) -> log(0) hatasını önlemek için +1 eklenebilir ama 
            # formül orijinalinde n=1'den başlar. (Burada n=0 iken tiklamalar da 0 oldugu icin else'e girer)
            delta = math.sqrt(3/2 * math.log(n + 1) / tiklamalar[i])
            
            ucb = ortalama + delta
        else:
            # Eğer bu ilan hiç seçilmediyse ona devasa bir UCB verip seçilmesini zorla
            ucb = 1e400 # Sonsuz
        
        if max_ucb < ucb:
            max_ucb = ucb
            ad = i

    # Seçilen ilanı kaydet
    secilenler.append(ad)
    tiklamalar[ad] += 1
    
    # --- ÖDÜL MEKANİZMASI (ENVIRONMENT) ---
    # Gerçek hayatta burası: "Kullanıcı tıkladı mı?" sorusunun cevabıdır.
    # Simülasyonda ise: df tablosunun n. satırına bakıyoruz.
    odul = df.values[n, ad] # 1 (Tıklandı) veya 0 (Tıklanmadı)
    
    oduller[ad] += odul
    toplam += odul

print(f"✅ Simülasyon Tamamlandı.")
print(f"💰 Toplam Ödül: {toplam}")

In [ ]:
# 3. SONUÇLARIN GÖRSELLEŞTİRİLMESİ
# ---------------------------------------------------
plt.figure(figsize=(10, 6))
plt.hist(secilenler)
plt.title('Sonuçlar (Hangi İlan Kaç Kez Seçildi?)')
plt.xlabel('İlanlar')
plt.ylabel('Seçilme Sayısı')
plt.grid(True, alpha=0.3)
plt.show()

# En çok seçilen ilanı bul
from collections import Counter
en_cok_secilen = Counter(secilenler).most_common(1)[0]
print(f"🏆 En Başarılı İlan: {en_cok_secilen[0]} (Seçilme Sayısı: {en_cok_secilen[1]})")